In [ ]:
import os
os.environ["GROQ_API_KEY"] = "GROQ_API"

In [3]:
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from groq import Groq

INDEX_PATH = "faiss_index.bin"
CHUNKS_PATH = "chunks.json"
TOP_K = 5  # how many chunks to retrieve

# Load once at import time (faster)
print("Loading models...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
index = faiss.read_index(INDEX_PATH)
with open(CHUNKS_PATH, "r") as f:
    chunks = json.load(f)

groq_client = Groq()  # reads GROQ_API_KEY from environment

def query(question: str) -> str:
    # Step 1: Embed the question
    q_embedding = embedder.encode([question])
    q_embedding = np.array(q_embedding).astype("float32")

    # Step 2: Search FAISS for top K similar chunks
    distances, indices = index.search(q_embedding, TOP_K)

    # Step 3: Build context from retrieved chunks
    context_parts = []
    for i, idx in enumerate(indices[0]):
        chunk = chunks[idx]
        context_parts.append(
            f"[Source: {chunk['source']}]\n{chunk['text']}"
        )
    context = "\n\n---\n\n".join(context_parts)

    # Step 4: Ask Groq LLM with the context
    prompt = f"""You are a helpful assistant. Answer the question based ONLY on the context below.
If the answer is not in the context, say "I couldn't find this in the documents."

Context:
{context}

Question: {question}

Answer:"""

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )

    return response.choices[0].message.content

if __name__ == "__main__":
    # Quick test
    answer = query("What is attention mechanism?")
    print(answer)

Loading models...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


I couldn't find a direct definition of the attention mechanism in the provided context. However, the context does describe the use of self-attention layers, multi-head attention, and different types of attention mechanisms, such as dot-product attention and additive attention, in the Transformer model. It also explains how these attention mechanisms are used in the encoder-decoder attention layers, self-attention layers in the encoder, and self-attention layers in the decoder. But a clear and concise definition of the attention mechanism itself is not provided.


In [4]:
answer = query("What problem does the Transformer architecture solve?")
print(answer)

The Transformer architecture solves the problem of sequential computation in traditional recurrent neural network (RNN) models, which precludes parallelization within training examples and becomes critical at longer sequence lengths. The Transformer model relies entirely on self-attention mechanisms to draw global dependencies between input and output, allowing for significantly more parallelization and reducing the computational constraints of RNNs.


In [5]:
answer = query("What datasets were used to evaluate RAG?")
print(answer)

The datasets used to evaluate RAG include Natural Questions, TriviaQA, WebQuestions, CuratedTrec, Jeopardy Question Generation, MS-MARCO, FEVER-3-way, and FEVER-2-way.


In [6]:
answer = query("What are the components of multi-head attention?")
print(answer)

The components of multi-head attention are:

1. Linear projections: The queries, keys, and values are linearly projected h times with different, learned linear projections to dk, dk, and dv dimensions, respectively.
2. Attention function: The attention function is applied to each of the projected queries, keys, and values in parallel, yielding dv-dimensional output values.
3. Concatenation: The output values from each attention function are concatenated.
4. Final projection: The concatenated output values are projected again using a parameter matrix WO to produce the final output values.

Mathematically, this can be represented as:

MultiHead(Q, K, V) = Concat(head1, ..., headh)WO

where headi = Attention(QWQ i, KW K i, V W V i)

and the projections are parameter matrices W Q i ∈Rdmodel×dk, W K i ∈Rdmodel×dk, W V i ∈Rdmodel×dv and W O ∈Rhdv×dmodel.


In [7]:
print(os.getcwd())

C:\Users\Tanmay\Research_assistant


In [10]:
import sys
print(sys.executable)

C:\Users\Tanmay\anaconda3\envs\ragmcp\python.exe


In [11]:
import sys
print(sys.executable)

C:\Users\Tanmay\anaconda3\envs\ragmcp\python.exe
